# Phase 1 — Data Understanding

## 1. Project Overview

This project uses the nine local CSV tables in the **Brazilian E-Commerce Public Dataset by Olist**. The business goal is to understand sales, products, delivery, and customer purchasing patterns so later phases can support business decisions with SQL Server and Power BI.

Phase 1 establishes what each table contains, its grain, candidate keys, missingness, and relationships. It includes descriptive statistics and data-quality diagnostics only. It does not clean data, develop business KPIs, create visualizations, or implement RFM, cohorts, SQL, Power BI, or machine learning.

Source attribution: Olist, Brazilian E-Commerce Public Dataset, [Kaggle publisher page](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). All findings below are calculated from the existing local download.

**How to run:** select the project `.venv` kernel and run all cells, in order. Paths work from the project root, notebook folder, or another folder inside this project. Original CSVs and loaded DataFrames are never overwritten. Reports are refreshed in `reports/`.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Locate the root from the notebook's current working directory.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [current_directory, *current_directory.parents]
     if (folder / "data" / "raw").is_dir() and (folder / "src" / "config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | NumPy: {np.__version__}")
print(f"Environment: {Path(sys.prefix).name}")
print(f"Project root: {PROJECT_ROOT}")

Python: 3.13.9 | pandas: 3.1.0 | NumPy: 2.5.4
Environment: .venv
Project root: E:\Projects CV\Sales Analytics & Business Intelligence


## 2. Load Datasets

Load every CSV into the `datasets` dictionary. IDs, category codes, and ZIP prefixes are read as strings so values such as `01037` keep their leading zeros. Timestamp fields remain strings; later cells parse separate Series for inspection. Other types are inferred by Pandas. Empty CSV fields use Pandas' standard missing-value recognition. This is a read configuration, not a cleaning step.

Full-file SHA-256 hashes are captured before loading and checked again at the end. Where available, the initialization manifest also verifies consistency with the original download.

In [2]:
DATASET_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

def file_hash(path):
    """Calculate a file digest without changing the file."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

datasets = {}
source_hashes = {}
for name, filename in DATASET_FILES.items():
    path = RAW_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    source_hashes[filename] = file_hash(path)
    columns = pd.read_csv(path, nrows=0).columns
    text_types = {
        column: "string" for column in columns
        if column.endswith("_id") or "zip_code_prefix" in column
        or column.startswith("product_category_name")
    }
    datasets[name] = pd.read_csv(path, dtype=text_types, low_memory=False)

manifest_path = PROJECT_ROOT / "data" / "DATASET_MANIFEST.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for entry in manifest:
        assert source_hashes[entry["filename"]] == entry["sha256"], entry["filename"]
print(f"Loaded {len(datasets)} datasets. Raw-file integrity checks passed.")

Loaded 9 datasets. Raw-file integrity checks passed.


## 3. Dataset Overview

The overview reports row counts, column counts, deep DataFrame memory usage (including the index), and source-file size. Memory is an estimate of loaded DataFrame storage, not total Python-process memory. Each table below also shows its column types and first five records. Identifier-like numbers and geographic coordinates should not be interpreted as business measures.

In [3]:
overview_rows = []
for name, frame in datasets.items():
    memory_bytes = int(frame.memory_usage(index=True, deep=True).sum())
    overview_rows.append({
        "dataset": name,
        "filename": DATASET_FILES[name],
        "rows": len(frame),
        "columns": len(frame.columns),
        "memory_bytes": memory_bytes,
        "memory_mib": memory_bytes / (1024 ** 2),
        "file_bytes": (RAW_DIR / DATASET_FILES[name]).stat().st_size,
        "column_names": json.dumps(frame.columns.tolist()),
        "column_dtypes": json.dumps({column: str(dtype) for column, dtype in frame.dtypes.items()}),
    })
dataset_overview = pd.DataFrame(overview_rows)
display(dataset_overview[["dataset", "rows", "columns", "memory_mib", "file_bytes"]])

,dataset,rows,columns,memory_mib,file_bytes
0,customers,99441,5,30.948784,9033957
1,geolocation,1000163,5,173.256764,61273883
2,orders,99441,8,52.937277,17654914
3,order_items,112650,7,40.501855,15438671
4,order_payments,103886,5,16.229413,5777138
5,order_reviews,99224,7,39.124777,14451670
6,products,32951,9,6.310526,2379446
7,sellers,3095,4,0.723823,174703
8,category_translation,71,2,0.008999,2613


In [4]:
for name, frame in datasets.items():
    display(Markdown(f"### {name}"))
    display(frame.dtypes.astype(str).rename("dtype").to_frame().rename_axis("column"))
    display(frame.head(5))
    memory_mib = frame.memory_usage(index=True, deep=True).sum() / (1024 ** 2)
    print(f"Rows: {len(frame):,} | Columns: {frame.shape[1]} | Deep memory: {memory_mib:.2f} MiB")

### customers

,dtype
column,
customer_id,string
customer_unique_id,string
customer_zip_code_prefix,string
customer_city,str
customer_state,str


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Rows: 99,441 | Columns: 5 | Deep memory: 30.95 MiB


### geolocation

,dtype
column,
geolocation_zip_code_prefix,string
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str


... (26 outputs left out)

### category_translation

,dtype
column,
product_category_name,string
product_category_name_english,string


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


Rows: 71 | Columns: 2 | Deep memory: 0.01 MiB


## 4. Descriptive Statistics

Use `describe()` for numerical columns and `describe(include=["object", "string"])` for categorical/text columns. Text summaries include non-missing counts, distinct values, and most frequent values; they also include IDs and unparsed timestamps for completeness. Tied modes may have multiple valid answers. Numerical summaries describe the source values and do not imply that every numeric field is a continuous measure. No rows are removed.

In [5]:
for name, frame in datasets.items():
    display(Markdown(f"### {name}"))
    numeric_columns = frame.select_dtypes(include=np.number).columns
    if len(numeric_columns):
        display(Markdown("**Numerical features**"))
        display(frame[numeric_columns].describe().T)
    text_columns = frame.select_dtypes(include=["object", "string"]).columns
    if len(text_columns):
        display(Markdown("**Categorical and text features**"))
        display(frame[text_columns].describe().T)

### customers

**Categorical and text features**

,count,unique,top,freq
customer_id,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_zip_code_prefix,99441,14994,22790,142
customer_city,99441,4119,sao paulo,15540
customer_state,99441,27,SP,41746


### geolocation

**Numerical features**

,count,mean,std,min,25%,50%,75%,max
geolocation_lat,1000163.0,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.979620,45.065933
geolocation_lng,1000163.0,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394


... (27 outputs left out)

,count,unique,top,freq
seller_id,3095,3095,3442f8959a84dea7ee197c632cb2df15,1
seller_zip_code_prefix,3095,2246,14940,49
seller_city,3095,611,sao paulo,694
seller_state,3095,23,SP,1849


### category_translation

**Categorical and text features**

,count,unique,top,freq
product_category_name,71,71,beleza_saude,1
product_category_name_english,71,71,health_beauty,1


## 5. Missing Values

For every column, count Pandas-recognized missing values and divide by that table's row count. Percentages use a 0–100 scale. Also inspect whitespace-only strings separately because they are not necessarily recognized as nulls. Missing review comments may be optional; missing delivery milestones require context from order status before any cleaning rule is chosen.

In [6]:
missing_rows = []
for name, frame in datasets.items():
    for column in frame.columns:
        missing_count = int(frame[column].isna().sum())
        whitespace_count = 0
        if pd.api.types.is_string_dtype(frame[column].dtype):
            whitespace_count = int(frame[column].str.fullmatch(r"\s*", na=False).sum())
        missing_rows.append({
            "dataset": name,
            "column": column,
            "dtype": str(frame[column].dtype),
            "total_rows": len(frame),
            "missing_count": missing_count,
            "missing_percentage": 100 * missing_count / len(frame) if len(frame) else np.nan,
            "whitespace_only_count": whitespace_count,
        })
missing_values_report = pd.DataFrame(missing_rows)
display(missing_values_report.sort_values(["missing_percentage", "dataset"], ascending=[False, True]))

,dataset,column,dtype,total_rows,missing_count,missing_percentage,whitespace_only_count
33,order_reviews,review_comment_title,str,99224,87656,88.341530,2
34,order_reviews,review_comment_message,str,99224,58247,58.702532,27
16,orders,order_delivered_customer_date,str,99441,2965,2.981668,0
38,products,product_category_name,string,32951,610,1.851234,0
39,products,product_name_lenght,float64,32951,610,1.851234,0
40,products,product_description_lenght,float64,32951,610,1.851234,0
41,products,product_photos_qty,float64,32951,610,1.851234,0
15,orders,order_delivered_carrier_date,str,99441,1783,1.793023,0
14,orders,order_approved_at,str,99441,160,0.160899,0
42,products,product_weight_g,float64,32951,2,0.006070,0


In [7]:
# Status context helps distinguish absent milestones from invalid values.
orders = datasets["orders"]
delivery_columns = [
    "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"
]
status_context = orders.groupby("order_status", dropna=False).size().rename("orders").to_frame()
for column in delivery_columns:
    status_context[f"{column}_missing"] = (
        orders[column].isna().groupby(orders["order_status"], dropna=False).sum()
    )
display(status_context)

,orders,order_approved_at_missing,order_delivered_carrier_date_missing,order_delivered_customer_date_missing
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


## 6. Duplicate Analysis

An **exact duplicate** repeats every source column. A **candidate primary key** should be unique and non-null at the table's grain. Report duplicate rows both as excess beyond the first occurrence and as all rows belonging to duplicate groups. No rows are dropped or edited.

`order_id` legitimately repeats in items, payments, and reviews. Item and payment sequences are checked together with `order_id`. `review_id` and `(review_id, order_id)` are separate hypotheses to test. ZIP prefixes describe areas and can map to many geolocation records; they are not assumed to be a geolocation primary key.

In [8]:
# True marks a primary-key hypothesis; False marks an intentionally repeatable identifier.
KEY_CHECKS = [
    ("customers", ["customer_id"], True),
    ("customers", ["customer_unique_id"], False),
    ("geolocation", ["geolocation_zip_code_prefix"], False),
    ("orders", ["order_id"], True),
    ("order_items", ["order_id", "order_item_id"], True),
    ("order_items", ["order_id"], False),
    ("order_payments", ["order_id", "payment_sequential"], True),
    ("order_payments", ["order_id"], False),
    ("order_reviews", ["review_id"], True),
    ("order_reviews", ["review_id", "order_id"], True),
    ("order_reviews", ["order_id"], False),
    ("products", ["product_id"], True),
    ("sellers", ["seller_id"], True),
    ("category_translation", ["product_category_name"], True),
]

duplicate_rows = []
for name, frame in datasets.items():
    duplicate_rows.append({
        "dataset": name, "check_type": "exact_rows", "key_columns": "ALL_COLUMNS",
        "candidate_primary_key": False, "total_rows": len(frame),
        "null_key_rows": np.nan,
        "distinct_non_null_keys": np.nan,
        "duplicate_rows_excess": int(frame.duplicated().sum()),
        "duplicate_rows_all": int(frame.duplicated(keep=False).sum()),
        "repeated_key_values": np.nan, "maximum_key_multiplicity": np.nan,
        "valid_primary_key": pd.NA,
    })

for name, columns, candidate in KEY_CHECKS:
    frame = datasets[name]
    null_mask = frame[columns].isna().any(axis=1)
    # Non-null keys are selected for diagnostics only; the source frame stays intact.
    key_counts = frame.loc[~null_mask, columns].value_counts()
    duplicate_rows.append({
        "dataset": name, "check_type": "key", "key_columns": " + ".join(columns),
        "candidate_primary_key": candidate, "total_rows": len(frame),
        "null_key_rows": int(null_mask.sum()),
        "distinct_non_null_keys": len(key_counts),
        "duplicate_rows_excess": int((key_counts - 1).sum()),
        "duplicate_rows_all": int(key_counts[key_counts > 1].sum()),
        "repeated_key_values": int((key_counts > 1).sum()),
        "maximum_key_multiplicity": int(key_counts.max()) if len(key_counts) else 0,
        "valid_primary_key": bool(not null_mask.any() and key_counts.le(1).all()),
    })
duplicate_summary = pd.DataFrame(duplicate_rows)
display(duplicate_summary)

,dataset,check_type,key_columns,candidate_primary_key,total_rows,null_key_rows,distinct_non_null_keys,duplicate_rows_excess,duplicate_rows_all,repeated_key_values,maximum_key_multiplicity,valid_primary_key
0,customers,exact_rows,ALL_COLUMNS,False,99441,NaN,NaN,0,0,NaN,NaN,<NA>
1,geolocation,exact_rows,ALL_COLUMNS,False,1000163,NaN,NaN,261831,390005,NaN,NaN,<NA>
2,orders,exact_rows,ALL_COLUMNS,False,99441,NaN,NaN,0,0,NaN,NaN,<NA>
3,order_items,exact_rows,ALL_COLUMNS,False,112650,NaN,NaN,0,0,NaN,NaN,<NA>
4,order_payments,exact_rows,ALL_COLUMNS,False,103886,NaN,NaN,0,0,NaN,NaN,<NA>
5,order_reviews,exact_rows,ALL_COLUMNS,False,99224,NaN,NaN,0,0,NaN,NaN,<NA>
6,products,exact_rows,ALL_COLUMNS,False,32951,NaN,NaN,0,0,NaN,NaN,<NA>
7,sellers,exact_rows,ALL_COLUMNS,False,3095,NaN,NaN,0,0,NaN,NaN,<NA>
8,category_translation,exact_rows,ALL_COLUMNS,False,71,NaN,NaN,0,0,NaN,NaN,<NA>
9,customers,key,customer_id,True,99441,0.0,99441.0,0,0,0.0,1.0,True


## 7. Data Relationships

`customer_id` identifies an order-linked customer record and is the join key in `orders`. `customer_unique_id` links customer records belonging to the same customer identity across purchases. It is the appropriate starting point for later repeat-purchase, RFM, and cohort work; it is not the primary key of the customer table. The checks below quantify the observed multiplicities rather than assuming one row equals one person.

Expected grains to validate:

| Table | Grain / candidate key |
|---|---|
| orders | One order, `order_id` |
| customers | One order-linked customer record, `customer_id` |
| order_items | One line item within an order, `(order_id, order_item_id)` |
| order_payments | One payment sequence within an order, `(order_id, payment_sequential)` |
| order_reviews | Review records; test `review_id` and `(review_id, order_id)` |
| products | One product, `product_id` |
| sellers | One seller, `seller_id` |
| category_translation | One Portuguese category-to-English mapping |
| geolocation | Geographic observations; ZIP prefix is a repeated lookup value |

Orders can have multiple items, payments, and reviews. Products and sellers can appear in many items. Categories can describe many products. Joining several detail tables directly by order can multiply rows and overstate later totals. Geolocation must not be joined as if ZIP prefixes were unique.

In [9]:
customers = datasets["customers"]
identity_counts = customers["customer_unique_id"].value_counts()
identity_summary = pd.DataFrame([{
    "customer_records": len(customers),
    "distinct_customer_id": customers["customer_id"].nunique(),
    "distinct_customer_unique_id": customers["customer_unique_id"].nunique(),
    "identities_with_multiple_customer_records": int((identity_counts > 1).sum()),
    "maximum_records_per_identity": int(identity_counts.max()),
}])
display(identity_summary)

,customer_records,distinct_customer_id,distinct_customer_unique_id,identities_with_multiple_customer_records,maximum_records_per_identity
0,99441,99441,96096,2997,17


In [10]:
# Each tuple is child table, child foreign key, parent table, parent lookup key.
RELATIONSHIPS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "category_translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]

relationship_rows = []
for child_name, child_key, parent_name, parent_key in RELATIONSHIPS:
    child = datasets[child_name][child_key]
    parent = datasets[parent_name][parent_key]
    parent_counts = parent.value_counts()
    child_counts = child.value_counts()
    parent_keys = parent_counts.index
    child_keys = child_counts.index
    matched = child.notna() & child.isin(parent_keys)
    unmatched = child.notna() & ~child.isin(parent_keys)
    missing_parent_keys = child_keys[~child_keys.isin(parent_keys)]
    matched_keys = child_keys[child_keys.isin(parent_keys)]
    parent_unique = bool(parent_counts.le(1).all() and parent.notna().all())
    parent_many = bool(parent_counts.reindex(matched_keys).gt(1).any())
    child_many = bool(child_counts.reindex(matched_keys).gt(1).any())
    if not len(matched_keys):
        cardinality = "no matched keys"
    else:
        cardinality = ("many" if parent_many else "one") + "-to-" + ("many" if child_many else "one")
    non_null_child_rows = int(child.notna().sum())
    relationship_rows.append({
        "child_dataset": child_name, "foreign_key": child_key,
        "parent_dataset": parent_name, "parent_key": parent_key,
        "child_rows": len(child), "null_foreign_key_rows": int(child.isna().sum()),
        "matched_child_rows": int(matched.sum()), "unmatched_child_rows": int(unmatched.sum()),
        "matched_percentage_non_null": 100 * matched.sum() / non_null_child_rows if non_null_child_rows else np.nan,
        "distinct_unmatched_foreign_keys": len(missing_parent_keys),
        "unmatched_key_examples": json.dumps(missing_parent_keys[:10].tolist()),
        "parent_key_unique_and_non_null": parent_unique,
        "parent_repeated_key_values": int(parent_counts.gt(1).sum()),
        "parent_null_key_rows": int(parent.isna().sum()),
        "parent_keys_without_child": int((~parent_keys.isin(child_keys)).sum()),
        "parent_rows_without_child": int(parent_counts.loc[~parent_keys.isin(child_keys)].sum()),
        "maximum_child_rows_per_matched_key": int(child_counts.reindex(matched_keys).max()) if len(matched_keys) else 0,
        "observed_parent_to_child": cardinality,
        "lookup_note": "unique parent lookup" if parent_unique else "repeated parent lookup; direct join can multiply rows",
    })
relationship_validation = pd.DataFrame(relationship_rows)
display(relationship_validation)

,child_dataset,foreign_key,parent_dataset,parent_key,child_rows,null_foreign_key_rows,matched_child_rows,unmatched_child_rows,matched_percentage_non_null,distinct_unmatched_foreign_keys,unmatched_key_examples,parent_key_unique_and_non_null,parent_repeated_key_values,parent_null_key_rows,parent_keys_without_child,parent_rows_without_child,maximum_child_rows_per_matched_key,observed_parent_to_child,lookup_note
0,orders,customer_id,customers,customer_id,99441,0,99441,0,100.000000,0,[],True,0,0,0,0,1,one-to-one,unique parent lookup
1,order_items,order_id,orders,order_id,112650,0,112650,0,100.000000,0,[],True,0,0,775,775,21,one-to-many,unique parent lookup
2,order_items,product_id,products,product_id,112650,0,112650,0,100.000000,0,[],True,0,0,0,0,527,one-to-many,unique parent lookup
3,order_items,seller_id,sellers,seller_id,112650,0,112650,0,100.000000,0,[],True,0,0,0,0,2033,one-to-many,unique parent lookup
4,order_payments,order_id,orders,order_id,103886,0,103886,0,100.000000,0,[],True,0,0,1,1,29,one-to-many,unique parent lookup
5,order_reviews,order_id,orders,order_id,99224,0,99224,0,100.000000,0,[],True,0,0,768,768,3,one-to-many,unique parent lookup
6,products,product_category_name,category_translation,product_category_name,32951,610,32328,13,99.959803,2,"[""portateis_cozinha_e_preparadores_de_alimento...",True,0,0,0,0,3029,one-to-many,unique parent lookup
7,customers,customer_zip_code_prefix,geolocation,geolocation_zip_code_prefix,99441,0,99163,278,99.720437,157,"[""70686"", ""72005"", ""71919"", ""73255"", ""72300"", ...",False,17972,0,4178,31864,142,many-to-many,repeated parent lookup; direct join can multip...
8,sellers,seller_zip_code_prefix,geolocation,geolocation_zip_code_prefix,3095,0,3088,7,99.773829,7,"[""82040"", ""91901"", ""72580"", ""02285"", ""07412"", ...",False,17972,0,16776,731347,49,many-to-many,repeated parent lookup; direct join can multip...


## 8. Date Columns

Identify date/timestamp fields from their actual names. Inspect source examples, string lengths, exact `YYYY-MM-DD HH:MM:SS` format matches, parse failures, and parsed ranges. Parsing uses a separate Series and never assigns converted values back to a source DataFrame. A failed parse is a diagnostic, not a replacement of the source value. No timezone is assumed.

In [11]:
date_rows = []
date_views = {}
for name, frame in datasets.items():
    date_columns = [column for column in frame.columns if "timestamp" in column or column.endswith(("_date", "_at"))]
    for column in date_columns:
        source = frame[column]
        text = source.astype("string")
        format_match = text.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", na=False)
        parsed = pd.to_datetime(source, format="%Y-%m-%d %H:%M:%S", errors="coerce")
        date_views[(name, column)] = parsed
        invalid = source.notna() & parsed.isna()
        date_rows.append({
            "dataset": name, "column": column, "source_dtype": str(source.dtype),
            "non_missing_count": int(source.notna().sum()),
            "missing_count": int(source.isna().sum()),
            "exact_format_mismatches": int((source.notna() & ~format_match).sum()),
            "parse_failures_non_missing": int(invalid.sum()),
            "minimum_parsed": str(parsed.min()), "maximum_parsed": str(parsed.max()),
            "source_examples": json.dumps(source.loc[source.notna()].head(3).tolist()),
            "string_lengths": json.dumps({int(length): int(count) for length, count in text.str.len().value_counts().items()}),
        })
date_profile = pd.DataFrame(date_rows)
display(date_profile)

,dataset,column,source_dtype,non_missing_count,missing_count,exact_format_mismatches,parse_failures_non_missing,minimum_parsed,maximum_parsed,source_examples,string_lengths
0,orders,order_purchase_timestamp,str,99441,0,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18,"[""2017-10-02 10:56:33"", ""2018-07-24 20:41:37"",...","{""19"": 99441}"
1,orders,order_approved_at,str,99281,160,0,0,2016-09-15 12:16:38,2018-09-03 17:40:06,"[""2017-10-02 11:07:15"", ""2018-07-26 03:24:27"",...","{""19"": 99281}"
2,orders,order_delivered_carrier_date,str,97658,1783,0,0,2016-10-08 10:34:01,2018-09-11 19:48:28,"[""2017-10-04 19:55:00"", ""2018-07-26 14:31:00"",...","{""19"": 97658}"
3,orders,order_delivered_customer_date,str,96476,2965,0,0,2016-10-11 13:46:32,2018-10-17 13:22:46,"[""2017-10-10 21:25:13"", ""2018-08-07 15:27:45"",...","{""19"": 96476}"
4,orders,order_estimated_delivery_date,str,99441,0,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00,"[""2017-10-18 00:00:00"", ""2018-08-13 00:00:00"",...","{""19"": 99441}"
5,order_items,shipping_limit_date,str,112650,0,0,0,2016-09-19 00:15:34,2020-04-09 22:35:08,"[""2017-09-19 09:45:35"", ""2017-05-03 11:05:13"",...","{""19"": 112650}"
6,order_reviews,review_creation_date,str,99224,0,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00,"[""2018-01-18 00:00:00"", ""2018-03-10 00:00:00"",...","{""19"": 99224}"
7,order_reviews,review_answer_timestamp,str,99224,0,0,0,2016-10-07 18:32:28,2018-10-29 12:27:35,"[""2018-01-18 21:46:59"", ""2018-03-11 03:05:13"",...","{""19"": 99224}"


In [12]:
# Compare timestamp pairs only where both values are present and parseable.
# Estimated delivery is a service estimate, not a strict chronological milestone.
date_pairs = [
    ("orders", "order_purchase_timestamp", "order_approved_at"),
    ("orders", "order_purchase_timestamp", "order_delivered_carrier_date"),
    ("orders", "order_purchase_timestamp", "order_delivered_customer_date"),
    ("orders", "order_approved_at", "order_delivered_carrier_date"),
    ("orders", "order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_reviews", "review_creation_date", "review_answer_timestamp"),
]
chronology_rows = []
for name, earlier, later in date_pairs:
    first = date_views[(name, earlier)]
    second = date_views[(name, later)]
    comparable = first.notna() & second.notna()
    chronology_rows.append({
        "dataset": name, "expected_earlier": earlier, "expected_later": later,
        "comparable_rows": int(comparable.sum()),
        "later_precedes_earlier_rows": int((comparable & (second < first)).sum()),
    })
chronology_report = pd.DataFrame(chronology_rows)
display(chronology_report)

,dataset,expected_earlier,expected_later,comparable_rows,later_precedes_earlier_rows
0,orders,order_purchase_timestamp,order_approved_at,99281,0
1,orders,order_purchase_timestamp,order_delivered_carrier_date,97658,166
2,orders,order_purchase_timestamp,order_delivered_customer_date,96476,0
3,orders,order_approved_at,order_delivered_carrier_date,97644,1359
4,orders,order_delivered_carrier_date,order_delivered_customer_date,96475,23
5,order_reviews,review_creation_date,review_answer_timestamp,99224,0


## 9. Initial Data Quality Findings

The summary below is assembled from the calculated results. Flags identify topics to investigate; they do not authorize deletion, imputation, deduplication, or correction. Additional checks cover non-positive product measurements, zero/negative financial amounts, and geographic coordinate ranges. Brazilian coordinate bounds would require a separately justified rule, so only global latitude/longitude bounds are checked here.

In [13]:
numeric_findings = []
numeric_checks = {
    "products": ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    "order_items": ["price", "freight_value"],
    "order_payments": ["payment_value", "payment_installments"],
}
for name, columns in numeric_checks.items():
    for column in columns:
        values = datasets[name][column]
        numeric_findings.append({
            "dataset": name, "column": column,
            "negative_count": int(values.lt(0).sum()), "zero_count": int(values.eq(0).sum()),
        })
numeric_quality = pd.DataFrame(numeric_findings)
display(numeric_quality)
geolocation = datasets["geolocation"]
invalid_coordinate_rows = int((
    geolocation["geolocation_lat"].abs().gt(90)
    | geolocation["geolocation_lng"].abs().gt(180)
).sum())
print(f"Geolocation rows outside global coordinate bounds: {invalid_coordinate_rows:,}")

,dataset,column,negative_count,zero_count
0,products,product_weight_g,0,4
1,products,product_length_cm,0,0
2,products,product_height_cm,0,0
3,products,product_width_cm,0,0
4,order_items,price,0,0
5,order_items,freight_value,0,383
6,order_payments,payment_value,0,9
7,order_payments,payment_installments,0,2


Geolocation rows outside global coordinate bounds: 0


In [14]:
def markdown_table(frame):
    """Render small report tables without installing an optional dependency."""
    def escape(value):
        return str(value).replace("|", "\|").replace("\n", " ")
    header = "| " + " | ".join(map(escape, frame.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(frame.columns)) + " |"
    rows = ["| " + " | ".join(escape(value) for value in row) + " |" for row in frame.itertuples(index=False, name=None)]
    return "\n".join([header, separator, *rows])

missing_findings = missing_values_report.loc[missing_values_report["missing_count"] > 0].copy()
missing_findings = missing_findings.sort_values("missing_percentage", ascending=False)
missing_display = missing_findings[["dataset", "column", "missing_count", "missing_percentage"]].copy()
missing_display["missing_percentage"] = missing_display["missing_percentage"].map(lambda value: f"{value:.2f}%")
exact_findings = duplicate_summary.loc[duplicate_summary["check_type"] == "exact_rows", [
    "dataset", "duplicate_rows_excess", "duplicate_rows_all"
]]
key_findings = duplicate_summary.loc[(duplicate_summary["check_type"] == "key") & duplicate_summary["candidate_primary_key"], [
    "dataset", "key_columns", "null_key_rows", "duplicate_rows_excess", "valid_primary_key"
]]
relationship_display = relationship_validation[[
    "child_dataset", "foreign_key", "parent_dataset", "null_foreign_key_rows",
    "unmatched_child_rows", "parent_keys_without_child", "observed_parent_to_child"
]]

summary_parts = [
    "# Phase 1 — Data Understanding Summary",
    "Computed from all nine local raw CSVs by `notebooks/01_data_understanding.ipynb`. Counts and diagnostics use the complete tables; head previews show five rows only. No cleaning or source-data modification was performed.",
    "## Dataset overview", markdown_table(dataset_overview[["dataset", "rows", "columns", "memory_mib"]].round(2)),
    "## Missing values", markdown_table(missing_display),
    f"Whitespace-only text fields: {int(missing_values_report['whitespace_only_count'].sum()):,}. Percentages use each source table's row count. Missing values use Pandas' default CSV null recognition.",
    "## Exact duplicates", markdown_table(exact_findings),
    "Excess counts exclude the first occurrence; all-duplicate counts include every row in duplicate groups. Repeated identifiers alone are not exact duplicate rows.",
    "## Candidate keys", markdown_table(key_findings),
    "Candidate keys are accepted only if unique and non-null. A failed hypothesis requires review, not automatic deletion. The complete duplicate CSV also includes intentionally repeatable identifiers.",
    "## Customer identity", markdown_table(identity_summary),
    "customer_id joins order-linked customer records to orders. customer_unique_id groups customer records across purchases; it must not be used as a unique key for the customer table.",
    "## Relationships", markdown_table(relationship_display),
    "Cardinality is parent-to-child for matched values in this download. Parent keys without children are coverage gaps, not automatically invalid records. Null child keys and unmatched non-null keys are reported separately. ZIP lookups are repeated observations; direct joins can multiply rows. Joining items, payments and reviews at order grain can also multiply rows.",
    "### Unmatched non-null foreign keys",
]
for row in relationship_validation.itertuples(index=False):
    if row.unmatched_child_rows:
        summary_parts.append(
            f"- {row.child_dataset}.{row.foreign_key} → {row.parent_dataset}.{row.parent_key}: "
            f"{row.unmatched_child_rows:,} rows, {row.distinct_unmatched_foreign_keys:,} distinct values; "
            f"examples: {row.unmatched_key_examples}."
        )
summary_parts += [
    "## Timestamp inspection", markdown_table(date_profile[[
        "dataset", "column", "missing_count", "exact_format_mismatches",
        "parse_failures_non_missing", "minimum_parsed", "maximum_parsed"
    ]]),
    "Dates were parsed into separate diagnostic Series with format %Y-%m-%d %H:%M:%S. Original DataFrame columns remain strings. Timezone information is absent from the inspected format; no timezone conversion was applied.",
    "### Missing milestones by order status", markdown_table(status_context.reset_index()),
    "### Chronological flags", markdown_table(chronology_report),
    "These are investigation flags, not proven corrections. Compare process semantics before selecting a Phase 2 rule.",
    "## Other initial numeric checks", markdown_table(numeric_quality),
    f"Geolocation rows outside global latitude/longitude bounds: {invalid_coordinate_rows:,}.",
    "Zero freight or payment amounts may be valid. Non-positive product measurements require investigation. Source column spellings such as product_name_lenght and product_description_lenght are retained.",
    "## Phase 2 investigation requirements",
]
for row in missing_findings.itertuples(index=False):
    summary_parts.append(f"- Define a missing-value rule for {row.dataset}.{row.column}: {row.missing_count:,} missing ({row.missing_percentage:.2f}%). Consider optional fields and order status before imputing.")
for row in exact_findings.itertuples(index=False):
    if row.duplicate_rows_excess:
        summary_parts.append(f"- Investigate {row.dataset}: {row.duplicate_rows_excess:,} excess exact duplicate rows; do not automatically deduplicate geographic observations.")
for row in key_findings.itertuples(index=False):
    if not row.valid_primary_key:
        summary_parts.append(f"- Review the {row.dataset} candidate key ({row.key_columns}): {row.duplicate_rows_excess:,} excess duplicate keys and {int(row.null_key_rows):,} null-key rows.")
for row in relationship_validation.itertuples(index=False):
    if row.unmatched_child_rows:
        summary_parts.append(f"- Plan unmatched-key handling for {row.child_dataset}.{row.foreign_key}: {row.unmatched_child_rows:,} rows do not match {row.parent_dataset}.")
for row in chronology_report.itertuples(index=False):
    if row.later_precedes_earlier_rows:
        summary_parts.append(f"- Investigate {row.later_precedes_earlier_rows:,} {row.dataset} rows where {row.expected_later} precedes {row.expected_earlier}.")
summary_parts += [
    "- Establish a documented geolocation lookup grain before joining ZIP prefixes; preserve leading zeros and investigate text variants before standardizing them.",
    "- Define valid ranges for product measurements and payment-related fields before making corrections; retain source spelling until any rename mapping is documented.",
    "- Preserve valid order-item and payment sequences and legitimate repeated customer identities. Resolve review-key assumptions before downstream modeling.",
    "## Execution and scope",
    "All cells reached report generation. Four CSV reports and this summary were generated and read back for validation. Raw-file SHA-256 hashes were checked before and after execution. No business metrics, visualizations, SQL, Power BI, machine learning, or Phase 2 transformations were implemented. Phase 1 is complete; wait for approval before Phase 2.",
]
phase_1_summary = "\n\n".join(summary_parts) + "\n"
display(Markdown(phase_1_summary))

<>:4: SyntaxWarning: invalid escape sequence '\|'
<>:4: SyntaxWarning: invalid escape sequence '\|'
C:\Users\hp\AppData\Local\Temp\ipykernel_19664\2550913762.py:4: SyntaxWarning: invalid escape sequence '\|'
  return str(value).replace("|", "\|").replace("\n", " ")


# Phase 1 — Data Understanding Summary

Computed from all nine local raw CSVs by `notebooks/01_data_understanding.ipynb`. Counts and diagnostics use the complete tables; head previews show five rows only. No cleaning or source-data modification was performed.

## Dataset overview

| dataset | rows | columns | memory_mib |
| --- | --- | --- | --- |
| customers | 99441 | 5 | 30.95 |
| geolocation | 1000163 | 5 | 173.26 |
| orders | 99441 | 8 | 52.94 |
| order_items | 112650 | 7 | 40.5 |
| order_payments | 103886 | 5 | 16.23 |
| order_reviews | 99224 | 7 | 39.12 |
| products | 32951 | 9 | 6.31 |
| sellers | 3095 | 4 | 0.72 |
| category_translation | 71 | 2 | 0.01 |

## Missing values

| dataset | column | missing_count | missing_percentage |
| --- | --- | --- | --- |
| order_reviews | review_comment_title | 87656 | 88.34% |
| order_reviews | review_comment_message | 58247 | 58.70% |
| orders | order_delivered_customer_date | 2965 | 2.98% |
| products | product_name_lenght | 610 | 1.85% |
| products | product_category_name | 610 | 1.85% |
| products | product_description_lenght | 610 | 1.85% |
| products | product_photos_qty | 610 | 1.85% |
| orders | order_delivered_carrier_date | 1783 | 1.79% |
| orders | order_approved_at | 160 | 0.16% |
| products | product_weight_g | 2 | 0.01% |
| products | product_length_cm | 2 | 0.01% |
| products | product_height_cm | 2 | 0.01% |
| products | product_width_cm | 2 | 0.01% |

Whitespace-only text fields: 29. Percentages use each source table's row count. Missing values use Pandas' default CSV null recognition.

## Exact duplicates

| dataset | duplicate_rows_excess | duplicate_rows_all |
| --- | --- | --- |
| customers | 0 | 0 |
| geolocation | 261831 | 390005 |
| orders | 0 | 0 |
| order_items | 0 | 0 |
| order_payments | 0 | 0 |
| order_reviews | 0 | 0 |
| products | 0 | 0 |
| sellers | 0 | 0 |
| category_translation | 0 | 0 |

Excess counts exclude the first occurrence; all-duplicate counts include every row in duplicate groups. Repeated identifiers alone are not exact duplicate rows.

## Candidate keys

| dataset | key_columns | null_key_rows | duplicate_rows_excess | valid_primary_key |
| --- | --- | --- | --- | --- |
| customers | customer_id | 0.0 | 0 | True |
| orders | order_id | 0.0 | 0 | True |
| order_items | order_id + order_item_id | 0.0 | 0 | True |
| order_payments | order_id + payment_sequential | 0.0 | 0 | True |
| order_reviews | review_id | 0.0 | 814 | False |
| order_reviews | review_id + order_id | 0.0 | 0 | True |
| products | product_id | 0.0 | 0 | True |
| sellers | seller_id | 0.0 | 0 | True |
| category_translation | product_category_name | 0.0 | 0 | True |

Candidate keys are accepted only if unique and non-null. A failed hypothesis requires review, not automatic deletion. The complete duplicate CSV also includes intentionally repeatable identifiers.

## Customer identity

| customer_records | distinct_customer_id | distinct_customer_unique_id | identities_with_multiple_customer_records | maximum_records_per_identity |
| --- | --- | --- | --- | --- |
| 99441 | 99441 | 96096 | 2997 | 17 |

customer_id joins order-linked customer records to orders. customer_unique_id groups customer records across purchases; it must not be used as a unique key for the customer table.

## Relationships

| child_dataset | foreign_key | parent_dataset | null_foreign_key_rows | unmatched_child_rows | parent_keys_without_child | observed_parent_to_child |
| --- | --- | --- | --- | --- | --- | --- |
| orders | customer_id | customers | 0 | 0 | 0 | one-to-one |
| order_items | order_id | orders | 0 | 0 | 775 | one-to-many |
| order_items | product_id | products | 0 | 0 | 0 | one-to-many |
| order_items | seller_id | sellers | 0 | 0 | 0 | one-to-many |
| order_payments | order_id | orders | 0 | 0 | 1 | one-to-many |
| order_reviews | order_id | orders | 0 | 0 | 768 | one-to-many |
| products | product_category_name | category_translation | 610 | 13 | 0 | one-to-many |
| customers | customer_zip_code_prefix | geolocation | 0 | 278 | 4178 | many-to-many |
| sellers | seller_zip_code_prefix | geolocation | 0 | 7 | 16776 | many-to-many |

Cardinality is parent-to-child for matched values in this download. Parent keys without children are coverage gaps, not automatically invalid records. Null child keys and unmatched non-null keys are reported separately. ZIP lookups are repeated observations; direct joins can multiply rows. Joining items, payments and reviews at order grain can also multiply rows.

### Unmatched non-null foreign keys

- products.product_category_name → category_translation.product_category_name: 13 rows, 2 distinct values; examples: ["portateis_cozinha_e_preparadores_de_alimentos", "pc_gamer"].

- customers.customer_zip_code_prefix → geolocation.geolocation_zip_code_prefix: 278 rows, 157 distinct values; examples: ["70686", "72005", "71919", "73255", "72300", "73369", "71884", "72002", "73401", "71676"].

- sellers.seller_zip_code_prefix → geolocation.geolocation_zip_code_prefix: 7 rows, 7 distinct values; examples: ["82040", "91901", "72580", "02285", "07412", "71551", "37708"].

## Timestamp inspection

| dataset | column | missing_count | exact_format_mismatches | parse_failures_non_missing | minimum_parsed | maximum_parsed |
| --- | --- | --- | --- | --- | --- | --- |
| orders | order_purchase_timestamp | 0 | 0 | 0 | 2016-09-04 21:15:19 | 2018-10-17 17:30:18 |
| orders | order_approved_at | 160 | 0 | 0 | 2016-09-15 12:16:38 | 2018-09-03 17:40:06 |
| orders | order_delivered_carrier_date | 1783 | 0 | 0 | 2016-10-08 10:34:01 | 2018-09-11 19:48:28 |
| orders | order_delivered_customer_date | 2965 | 0 | 0 | 2016-10-11 13:46:32 | 2018-10-17 13:22:46 |
| orders | order_estimated_delivery_date | 0 | 0 | 0 | 2016-09-30 00:00:00 | 2018-11-12 00:00:00 |
| order_items | shipping_limit_date | 0 | 0 | 0 | 2016-09-19 00:15:34 | 2020-04-09 22:35:08 |
| order_reviews | review_creation_date | 0 | 0 | 0 | 2016-10-02 00:00:00 | 2018-08-31 00:00:00 |
| order_reviews | review_answer_timestamp | 0 | 0 | 0 | 2016-10-07 18:32:28 | 2018-10-29 12:27:35 |

Dates were parsed into separate diagnostic Series with format %Y-%m-%d %H:%M:%S. Original DataFrame columns remain strings. Timezone information is absent from the inspected format; no timezone conversion was applied.

### Missing milestones by order status

| order_status | orders | order_approved_at_missing | order_delivered_carrier_date_missing | order_delivered_customer_date_missing |
| --- | --- | --- | --- | --- |
| approved | 2 | 0 | 2 | 2 |
| canceled | 625 | 141 | 550 | 619 |
| created | 5 | 5 | 5 | 5 |
| delivered | 96478 | 14 | 2 | 8 |
| invoiced | 314 | 0 | 314 | 314 |
| processing | 301 | 0 | 301 | 301 |
| shipped | 1107 | 0 | 0 | 1107 |
| unavailable | 609 | 0 | 609 | 609 |

### Chronological flags

| dataset | expected_earlier | expected_later | comparable_rows | later_precedes_earlier_rows |
| --- | --- | --- | --- | --- |
| orders | order_purchase_timestamp | order_approved_at | 99281 | 0 |
| orders | order_purchase_timestamp | order_delivered_carrier_date | 97658 | 166 |
| orders | order_purchase_timestamp | order_delivered_customer_date | 96476 | 0 |
| orders | order_approved_at | order_delivered_carrier_date | 97644 | 1359 |
| orders | order_delivered_carrier_date | order_delivered_customer_date | 96475 | 23 |
| order_reviews | review_creation_date | review_answer_timestamp | 99224 | 0 |

These are investigation flags, not proven corrections. Compare process semantics before selecting a Phase 2 rule.

## Other initial numeric checks

| dataset | column | negative_count | zero_count |
| --- | --- | --- | --- |
| products | product_weight_g | 0 | 4 |
| products | product_length_cm | 0 | 0 |
| products | product_height_cm | 0 | 0 |
| products | product_width_cm | 0 | 0 |
| order_items | price | 0 | 0 |
| order_items | freight_value | 0 | 383 |
| order_payments | payment_value | 0 | 9 |
| order_payments | payment_installments | 0 | 2 |

Geolocation rows outside global latitude/longitude bounds: 0.

Zero freight or payment amounts may be valid. Non-positive product measurements require investigation. Source column spellings such as product_name_lenght and product_description_lenght are retained.

## Phase 2 investigation requirements

- Define a missing-value rule for order_reviews.review_comment_title: 87,656 missing (88.34%). Consider optional fields and order status before imputing.

- Define a missing-value rule for order_reviews.review_comment_message: 58,247 missing (58.70%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_delivered_customer_date: 2,965 missing (2.98%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_name_lenght: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_category_name: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_description_lenght: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_photos_qty: 610 missing (1.85%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_delivered_carrier_date: 1,783 missing (1.79%). Consider optional fields and order status before imputing.

- Define a missing-value rule for orders.order_approved_at: 160 missing (0.16%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_weight_g: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_length_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_height_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Define a missing-value rule for products.product_width_cm: 2 missing (0.01%). Consider optional fields and order status before imputing.

- Investigate geolocation: 261,831 excess exact duplicate rows; do not automatically deduplicate geographic observations.

- Review the order_reviews candidate key (review_id): 814 excess duplicate keys and 0 null-key rows.

- Plan unmatched-key handling for products.product_category_name: 13 rows do not match category_translation.

- Plan unmatched-key handling for customers.customer_zip_code_prefix: 278 rows do not match geolocation.

- Plan unmatched-key handling for sellers.seller_zip_code_prefix: 7 rows do not match geolocation.

- Investigate 166 orders rows where order_delivered_carrier_date precedes order_purchase_timestamp.

- Investigate 1,359 orders rows where order_delivered_carrier_date precedes order_approved_at.

- Investigate 23 orders rows where order_delivered_customer_date precedes order_delivered_carrier_date.

- Establish a documented geolocation lookup grain before joining ZIP prefixes; preserve leading zeros and investigate text variants before standardizing them.

- Define valid ranges for product measurements and payment-related fields before making corrections; retain source spelling until any rename mapping is documented.

- Preserve valid order-item and payment sequences and legitimate repeated customer identities. Resolve review-key assumptions before downstream modeling.

## Execution and scope

All cells reached report generation. Four CSV reports and this summary were generated and read back for validation. Raw-file SHA-256 hashes were checked before and after execution. No business metrics, visualizations, SQL, Power BI, machine learning, or Phase 2 transformations were implemented. Phase 1 is complete; wait for approval before Phase 2.


In [15]:
# Export diagnostics only. None of the source DataFrames are written to disk.
csv_reports = {
    "dataset_overview.csv": dataset_overview,
    "missing_values_report.csv": missing_values_report,
    "duplicate_summary.csv": duplicate_summary,
    "relationship_validation.csv": relationship_validation,
}
for filename, report in csv_reports.items():
    path = REPORTS_DIR / filename
    report.to_csv(path, index=False, encoding="utf-8")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == report.shape, f"Report shape mismatch: {filename}"

# Reconcile diagnostic coverage and denominators before marking execution complete.
assert len(dataset_overview) == len(DATASET_FILES) == 9
assert len(missing_values_report) == sum(frame.shape[1] for frame in datasets.values())
assert missing_values_report["missing_count"].le(missing_values_report["total_rows"]).all()
assert len(duplicate_summary.loc[duplicate_summary["check_type"] == "exact_rows"]) == 9
assert len(relationship_validation) == len(RELATIONSHIPS)
assert (
    relationship_validation["null_foreign_key_rows"]
    + relationship_validation["matched_child_rows"]
    + relationship_validation["unmatched_child_rows"]
).equals(relationship_validation["child_rows"])
for filename, original_hash in source_hashes.items():
    assert file_hash(RAW_DIR / filename) == original_hash, f"Raw file changed: {filename}"
    original_path = PROJECT_ROOT / filename
    if original_path.is_file():
        assert file_hash(original_path) == original_hash, f"Root original differs: {filename}"

summary_path = REPORTS_DIR / "phase_1_summary.md"
summary_path.write_text(phase_1_summary, encoding="utf-8")
assert summary_path.read_text(encoding="utf-8") == phase_1_summary
for path in [*(REPORTS_DIR / filename for filename in csv_reports), summary_path]:
    print(f"Verified: {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")
print("Phase 1 complete. Raw data unchanged. Stop here and wait for approval before Phase 2.")

Verified: reports\dataset_overview.csv (3,830 bytes)
Verified: reports\missing_values_report.csv (2,908 bytes)
Verified: reports\duplicate_summary.csv (1,830 bytes)
Verified: reports\relationship_validation.csv (1,971 bytes)
Verified: reports\phase_1_summary.md (12,118 bytes)
Phase 1 complete. Raw data unchanged. Stop here and wait for approval before Phase 2.
